# 🚀 GRACE: Tự Host Server LLM Mã Nguồn Mở Trên Kaggle GPU (0 USD)

Notebook này được thiết kế theo kiến trúc **Plug-in / Modular Model Registry** thuộc **Bước 1** của Đề tài GRACE.
Mục tiêu: Tự host các mô hình mã nguồn mở (`Qwen2.5-Coder-7B`, `14B`, `32B`, `Qwen3-Coder-30B`) trên tài nguyên **Kaggle GPU miễn phí 100%** (2× Tesla T4 16GB = 32GB VRAM hoặc 1× P100 16GB VRAM).

### Tính năng cốt lõi:
1. **Plug-and-Play Registry**: Đổi model và engine linh hoạt chỉ bằng cấu hình `SELECTED_MODEL` và `SELECTED_ENGINE` (`llamacpp` hoặc `vllm`).
2. **Hỗ trợ 2×T4 Layer Splitting / Tensor Parallelism**: Chạy mượt mà model 32B (GGUF Q4_K_M ~20GB hoặc AWQ) phân bổ trên 2 card T4.
3. **OpenAI API Chuẩn**: Endpoint tương thích 100% trả về `logprobs=True, top_logprobs=5` để kết nối thẳng với `smoke_test_backend.py` và `run_matrix.py` của Thầy.
4. **Kiểm thử Kép**: Cung cấp cả tế bào kiểm thử bằng Python Client và lệnh `curl` CLI tại chỗ.
5. **Public Tunnel**: Mở đường hầm HTTPS miễn phí (qua Cloudflared / Ngrok) cho máy trạm bên ngoài kết nối vào.

## Bước 1: Phát hiện & Kiểm tra Phần cứng GPU

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"GPU Count: {gpu_count}")
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

## Bước 2: Cài đặt Môi trường & Engine (llama-server / llama-cpp-python / vLLM / Tunnel Tools)

In [ ]:
# 1. Cài đặt các thư viện Python cơ bản
!pip install -q huggingface_hub openai pyyaml requests

# 2. Cài đặt llama-cpp-python có hỗ trợ CUDA prebuilt wheels (tương thích CUDA 12.x trên Kaggle)
!pip install -q llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 || pip install -q llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

# 3. Tải Cloudflared binary để tạo HTTPS tunnel công khai (0 USD, không cần đăng ký tài khoản)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 4. Cài đặt vLLM nếu chọn engine vllm (tùy chọn)
# !pip install -q vllm

print("✅ Hoàn tất cài đặt các công cụ nền tảng!")

## Bước 3: Cấu hình Plug-in Model Registry
Chọn mô hình và engine bạn muốn host từ danh mục dưới đây:

In [ ]:
# DANH MỤC PLUG-IN MODEL:
# 1. 'qwen2.5-coder-7b-instruct'    (~5.5 GB VRAM, chạy được trên 1xP100 hoặc 2xT4)
# 2. 'qwen2.5-coder-14b-instruct'   (~9.8 GB VRAM, chạy được trên 1xP100 hoặc 2xT4)
# 3. 'qwen2.5-coder-32b-instruct'   (~20.5 GB VRAM, khuyến nghị 2xT4 layer-split hoặc AWQ)
# 4. 'qwen3-coder-30b-a3b-instruct' (~18.5 GB VRAM, MoE Baseline của Thầy, 2xT4 layer-split)

SELECTED_MODEL = "qwen3-coder-30b-a3b-instruct"   # <-- ĐÃ CHUYỂN SANG QWEN3 MOE 30B THEO ĐỀ CƯƠNG
SELECTED_ENGINE = "llamacpp"                       # Lựa chọn: 'llamacpp' | 'vllm'
SERVER_PORT = 8080

MODEL_REGISTRY = {
    "qwen2.5-coder-7b-instruct": {
        "name": "Qwen2.5-Coder-7B-Instruct",
        "repo": "Qwen/Qwen2.5-Coder-7B-Instruct-GGUF",
        "file": "qwen2.5-coder-7b-instruct-q4_k_m.gguf",
        "awq_repo": "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
        "alias": "qwen2.5-coder-7b-instruct",
        "split_mode": "none",
        "tensor_parallel": 1,
    },
    "qwen2.5-coder-14b-instruct": {
        "name": "Qwen2.5-Coder-14B-Instruct",
        "repo": "Qwen/Qwen2.5-Coder-14B-Instruct-GGUF",
        "file": "qwen2.5-coder-14b-instruct-q4_k_m.gguf",
        "awq_repo": "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ",
        "alias": "qwen2.5-coder-14b-instruct",
        "split_mode": "none",
        "tensor_parallel": 1,
    },
    "qwen2.5-coder-32b-instruct": {
        "name": "Qwen2.5-Coder-32B-Instruct",
        "repo": "Qwen/Qwen2.5-Coder-32B-Instruct-GGUF",
        "file": "qwen2.5-coder-32b-instruct-q4_k_m.gguf",
        "awq_repo": "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ",
        "alias": "qwen2.5-coder-32b-instruct",
        "split_mode": "layer",
        "tensor_parallel": 2,
    },
    "qwen3-coder-30b-a3b-instruct": {
        "name": "Qwen3-Coder-30B-A3B-Instruct (MoE)",
        "repo": "unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF",
        "file": "Qwen3-Coder-30B-A3B-Instruct-UD-Q4_K_XL.gguf",
        "awq_repo": "Qwen/Qwen3-Coder-30B-A3B-Instruct",
        "alias": "qwen3-coder-30b-q4",
        "split_mode": "layer",
        "tensor_parallel": 2,
    },
}

assert SELECTED_MODEL in MODEL_REGISTRY, f"Model {SELECTED_MODEL} chưa được định nghĩa trong registry!"
target_cfg = MODEL_REGISTRY[SELECTED_MODEL]
print(f"[*] Đã chọn Model:  {target_cfg['name']}")
print(f"[*] Engine:         {SELECTED_ENGINE}")
if SELECTED_ENGINE == 'llamacpp':
    print(f"[*] GGUF Source:    {target_cfg['repo']}/{target_cfg['file']}")
else:
    print(f"[*] vLLM Source:    {target_cfg['awq_repo']}")


## Bước 4: Tải Trọng số Mô hình (GGUF Q4 hoặc HF Repo)

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

# 1. Smart Discovery: Quét toàn bộ các Kaggle Datasets / Models đã mount trong /kaggle/input
found_in_input = None
input_dir = Path("/kaggle/input")
if input_dir.exists():
    for p in input_dir.rglob("*.gguf"):
        fname = p.name.lower()
        target_fname = target_cfg["file"].lower()
        # Khớp chính xác tên file hoặc các biến thể Q4_K_XL / IQ4 của Qwen3 Coder 30B
        if target_fname in fname or fname in target_fname or ("qwen3" in fname and "30b" in fname and "q4" in fname):
            found_in_input = p
            break

if found_in_input and found_in_input.exists():
    model_path = found_in_input
    print(f"✅ TÌM THẤY MÔ HÌNH SẴN CÓ TRONG KAGGLE INPUT (0s tải!):")
    print(f"   Đường dẫn: {model_path}")
    print(f"   Dung lượng:  {model_path.stat().st_size / (1024**3):.2f} GB")
elif SELECTED_ENGINE == "llamacpp":
    # 2. Fallback nếu chưa mount dataset: Tải về /tmp/models để không làm nặng thư mục output
    models_dir = Path("/tmp/models")
    models_dir.mkdir(parents=True, exist_ok=True)
    model_path = models_dir / target_cfg["file"]
    if model_path.exists():
        print(f"[*] Mô hình đã có sẵn trong cache /tmp: {model_path} ({model_path.stat().st_size / (1024**3):.2f} GB)")
    else:
        print(f"[*] Chưa thấy dataset trong input. Đang tải trọng số từ Hugging Face Hub: {target_cfg['repo']}...")
        model_path = Path(hf_hub_download(
            repo_id=target_cfg["repo"],
            filename=target_cfg["file"],
            local_dir=str(models_dir),
        ))
        print(f"✅ Tải thành công! Đường dẫn file: {model_path}")
else:
    print(f"[*] Engine vLLM sẽ tự động tải hoặc cache từ HF Repo: {target_cfg['awq_repo']}")
    model_path = target_cfg["awq_repo"]


## Bước 5: Khởi Động Server Nền (llama-server / llama_cpp.server / vLLM)

In [ ]:
import torch
import shutil
import subprocess
import sys
import time
import urllib.request

log_path = "/kaggle/working/server.log"
log_file = open(log_path, "w")

if SELECTED_ENGINE == "llamacpp":
    if shutil.which("llama-server"):
        cmd = [
            "llama-server",
            "-m", str(model_path),
            "--alias", target_cfg["alias"],
            "--n-gpu-layers", "99",
            "--ctx-size", "4096",
            "--parallel", "1",
            "--host", "0.0.0.0",
            "--port", str(SERVER_PORT),
            "--seed", "42",
        ]
        if target_cfg.get("split_mode") == "layer" and torch.cuda.device_count() >= 2:
            cmd.extend(["--split-mode", "layer"])
    else:
        cmd = [
            sys.executable, "-m", "llama_cpp.server",
            "--model", str(model_path),
            "--model_alias", target_cfg["alias"],
            "--n_gpu_layers", "99",
            "--n_ctx", "4096",
            "--use_mlock", "False",
            "--host", "0.0.0.0",
            "--port", str(SERVER_PORT),
        ]
elif SELECTED_ENGINE == "vllm":
    tp = target_cfg.get("tensor_parallel", 1) if torch.cuda.device_count() >= 2 else 1
    cmd = [
        "vllm", "serve", str(model_path),
        "--host", "0.0.0.0",
        "--port", str(SERVER_PORT),
        "--tensor-parallel-size", str(tp),
        "--quantization", "awq",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enforce-eager",
    ]
else:
    raise ValueError(f"Engine {SELECTED_ENGINE} không được hỗ trợ!")

print(f"[*] Khởi động tiến trình server: {' '.join(cmd)}")
server_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
print(f"[*] Server đang chạy với PID: {server_process.pid}")

print("[*] Đang chờ server nạp mô hình vào GPU (khoảng 120-180 giây cho model MoE 30B)... và kiểm tra cổng:")
ready = False
# Chờ tối đa 120 lần x 2s = 240s (4 phút) để đảm bảo model 16.5GB nạp đủ vào 2x GPU
for attempt in range(120):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{SERVER_PORT}/v1/models", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                print(f"\n🟢 SERVER ĐÃ SẴN SÀNG PHỤC VỤ TRUY VẤN! (Thời gian nạp: ~{attempt*2}s)")
                break
    except Exception:
        time.sleep(2)
        if (attempt + 1) % 10 == 0:
            print(f"[{attempt*2}s]", end="", flush=True)
        else:
            print(".", end="", flush=True)

if not ready:
    print("\n[!] LỖI: Server chưa phản hồi /v1/models sau 240s. Kiểm tra log:")
    !tail -n 50 /kaggle/working/server.log

assert ready, "Server chưa khởi động thành công sau 240 giây!"


## Bước 6: Kiểm Thử Tại Chỗ Bằng Python Client (Smoke Test)
Kiểm tra xem server có trả về `top_logprobs` chứa token `0` hoặc `1` đúng chuẩn yêu cầu của Thầy hay không.

In [ ]:
import json
import math
import urllib.request

test_prompt = (
    "You are an automated code security auditor. Answer 1 if vulnerable, 0 if safe.\n\n"
    "Function code:\n"
    "void process(char *input) {\n"
    "    char buffer[16];\n"
    "    strcpy(buffer, input);\n"
    "}\n\n"
    "Is this code vulnerable? Answer only 0 or 1:"
)

payload = {
    "model": target_cfg["alias"],
    "messages": [{"role": "user", "content": test_prompt}],
    "temperature": 0.0,
    "max_tokens": 1,
    "logprobs": True,
    "top_logprobs": 5,
    "seed": 42,
}

req = urllib.request.Request(
    f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(req, timeout=30) as resp:
    res_data = json.loads(resp.read().decode("utf-8"))

# Trích xuất logprobs của token đầu tiên
choice = res_data["choices"][0]
raw_pred = choice["message"]["content"]
top_lps = choice.get("logprobs", {}).get("content", [{}])[0].get("top_logprobs", [])

token_dict = {item["token"].strip(): item["logprob"] for item in top_lps}
has_0 = "0" in token_dict
has_1 = "1" in token_dict
p0 = math.exp(token_dict["0"]) if has_0 else 0.0
p1 = math.exp(token_dict["1"]) if has_1 else 0.0
s = p1 / (p0 + p1) if (p0 + p1) > 0 else float("nan")

print("=" * 60)
print("       KẾT QUẢ SMOKE TEST TẠI CHỖ (PYTHON CLIENT)")
print("=" * 60)
print(f"  Dự đoán thô (raw):      '{raw_pred}'")
print(f"  Top tokens nhận được:   {list(token_dict.keys())}")
print(f"  Có token '0' (Safe):    {has_0} (xác suất p0 = {p0:.4f})")
print(f"  Có token '1' (Vuln):    {has_1} (xác suất p1 = {p1:.4f})")
print(f"  Điểm xác suất liên tục: s = {s:.4f}")
print("=" * 60)

if has_0 or has_1:
    print("🎉 CHÚC MỪNG: Server trả đầy đủ top_logprobs hợp lệ và sẵn sàng kết nối!")
else:
    print("⚠️ Cảnh báo: Token 0 hoặc 1 không xuất hiện trong top-5 logprobs.")

## Bước 7: Kiểm Thử Tại Chỗ Bằng Lệnh `curl` CLI
Gửi truy vấn trực tiếp bằng công cụ dòng lệnh `curl` để xác nhận endpoint OpenAI tương thích 100%:

In [ ]:
# Sử dụng alias mô hình được chọn tự động từ cấu hình
import json, subprocess
curl_payload = json.dumps({
    "model": target_cfg["alias"],
    "messages": [{"role": "user", "content": "Is this code vulnerable? Answer only 0 or 1:\nvoid test() { char buf[8]; strcpy(buf, \"1234567890\"); }"}],
    "temperature": 0.0,
    "max_tokens": 1,
    "logprobs": True,
    "top_logprobs": 5
})
res = subprocess.run(["curl", "-s", "-X", "POST", f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions",
                      "-H", "Content-Type: application/json", "-d", curl_payload],
                     capture_output=True, text=True)
try:
    parsed = json.loads(res.stdout)
    print("CURL CLI TEST PASSED:")
    print(json.dumps(parsed, indent=2))
except Exception:
    print("CURL OUTPUT:", res.stdout)


## Bước 8: Mở Đường Hầm HTTPS Công Khai (Cloudflared Tunnel)
Bước này cho phép bạn truy cập server từ máy tính cá nhân hoặc máy trạm bên ngoài mà không cần tài khoản.

In [ ]:
import re
import subprocess
import time

# Khởi động cloudflared tunnel trỏ tới SERVER_PORT
tunnel_log = open("/kaggle/working/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

print("[*] Đang khởi tạo Cloudflare Tunnel...")
public_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists("/kaggle/working/tunnel.log"):
        content = open("/kaggle/working/tunnel.log").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("\n" + "=" * 70)
    print("🌐 ĐƯỜNG DẪN PUBLIC API SERVER CỦA BẠN ĐÃ SẴN SÀNG:")
    print(f"   URL: {public_url}/v1")
    print("=" * 70)
else:
    print("[!] Chưa bắt được URL Cloudflared. Kiểm tra log:")
    !cat /kaggle/working/tunnel.log

## Bước 9: Kết Xuất File Cấu Hình Backend cho `grace_diag_kit`
Sao chép đoạn nội dung YAML dưới đây vào file `grace_diag_kit/configs/backend_kaggle.yaml` trên máy trạm của bạn để chạy `smoke_test_backend.py` và `run_matrix.py`!

In [ ]:
endpoint_url = f"{public_url}/v1" if public_url else f"http://127.0.0.1:{SERVER_PORT}/v1"

backend_yaml_text = f"""# Cấu hình backend kết nối tới Kaggle Server tự host (0 USD)
backend:
  kind: {SELECTED_ENGINE}
  base_url: {endpoint_url}
  model: {target_cfg['alias']}
  api_key: EMPTY
  temperature: 0.0
  max_tokens: 1
  top_logprobs: 5
  seed: 42
  guided_choice: false
  timeout_s: 300
  system_prompt: null
"""

with open("/kaggle/working/backend_kaggle.yaml", "w") as f:
    f.write(backend_yaml_text)

print("=" * 70)
print("📄 NỘI DUNG FILE backend_kaggle.yaml ĐỂ DÙNG TRÊN MÁY LOCAL:")
print("=" * 70)
print(backend_yaml_text)
print("=" * 70)
print("\nĐể kiểm thử từ máy local, chạy lệnh:")
print("python grace_diag_kit/scripts/smoke_test_backend.py --backend grace_diag_kit/configs/backend_kaggle.yaml --prompts <file_prompts.jsonl> --n 20")